### Problem 1

In [ ]:
import numpy as np
from scipy import stats
from scipy.stats import chisquare


def rng(m=2**32, a=1103515245, c=12345):
    rng.current = (a * rng.current + c) % m
    return rng.current / m

# setting the seed
rng.current = 1

samples = [rng() for _ in range(1000)]


def serial_test(sequence, k, bins=10):
    sequence = np.array(sequence)
    n = len(sequence)

    # Number of tuples
    m = n // k
    trimmed = sequence[:m * k]

    # Reshape into k-dimensional points
    points = trimmed.reshape((m, k))

    # Create multidimensional histogram
    hist, edges = np.histogramdd(points, bins=[bins]*k, range=[[0,1]]*k)

    observed = hist.flatten()
    
    # Expected frequency (uniform)
    expected = np.full_like(observed, fill_value=m / (bins**k), dtype=float)

    chi2, p_value = chisquare(observed, expected)

    return chisquare(observed, expected)

serial_test(samples, 2, bins=15)

Power_divergenceResult(statistic=np.float64(255.09999999999997), pvalue=np.float64(0.07528556709734432))

### Problem 2

In [17]:
import numpy as np
import itertools

def get_permutation_index(block):
    ranks = np.argsort(block)
    return tuple(ranks)


def permutation_test(sequence, d=3):
    sequence = np.array(sequence)
    n = len(sequence) // d

    # Обрезаем лишнее
    blocks = sequence[:n*d].reshape(n, d)

    # Все возможные перестановки
    perms = list(itertools.permutations(range(d)))
    perm_index = {p: i for i, p in enumerate(perms)}

    counts = np.zeros(len(perms))

    # Считаем
    for block in blocks:
        p = get_permutation_index(block)
        counts[perm_index[p]] += 1

    # Ожидаемое значение
    expected = n / len(perms)

    chi2 = np.sum((counts - expected)**2 / expected)

    return chi2, counts

permutation_test(samples, 3)

(np.float64(4.8558558558558556), array([49., 62., 61., 60., 57., 44.]))

### Problem 3

In [22]:
n = 1000

def rng_bad(m=2**31, a=65539, c=0):
    rng_bad.current = (a * rng_bad.current + c) % m
    return rng_bad.current / m
rng_bad.current = 1


def generate_samples(rng_func, n):
    return np.array([rng_func() for _ in range(n)])
    

sample_bad = generate_samples(rng_bad, n)

rng.current = 1
sample_good = generate_samples(rng, n)

chi2_bad, p_bad = serial_test(sample_bad, k=2)
chi2_good, p_good = serial_test(sample_good, k=2)

print("BAD RNG  -> chi2:", chi2_bad, "p-value:", p_bad)
print("GOOD RNG -> chi2:", chi2_good, "p-value:", p_good)

BAD RNG  -> chi2: 127.60000000000001 p-value: 0.02802262075660813
GOOD RNG -> chi2: 114.39999999999999 p-value: 0.1380441495447643


### Problem 4

In [ ]:
digits = np.arange(10)
weights = np.array([0.12, 0.3, 0.167, 0.24, 0.31, 0.54, 0.111, 0.02, 0.001, 0.2])

# нормализация
weights = weights / weights.sum()

# CDF
cdf = np.cumsum(weights)

def sample_one():
    u = np.random.rand()
    return np.searchsorted(cdf, u)

# выборка
sample = np.array([sample_one() for _ in range(1000)])

plt.hist(sample, bins=np.arange(11)-0.5)
plt.xticks(range(10))
plt.title("Histogram of sample")
plt.show()